<a href="https://colab.research.google.com/github/demsaid400-cpu/DI-BOOTCAMP/blob/main/Daily_Challenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "bigscience/bloomz-560m"
tokenizer = AutoTokenizer.from_pretrained(model_name)
foundation_model = AutoModelForCausalLM.from_pretrained(model_name)

data = load_dataset("Abirate/english_quotes", split="train[:10%]")

# Modification : Ajout du padding, de la troncature et suppression des colonnes non nécessaires
data = data.map(
    lambda samples: tokenizer(samples["quote"], padding="max_length", truncation=True, max_length=128),
    batched=True,
    remove_columns=data.column_names
)

train_sample = data.select(range(5))
display(train_sample)

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

Map:   0%|          | 0/251 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 5
})

In [9]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=1,
    lora_alpha=1,
    target_modules=["query_key_value"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [12]:
# Upgrade torchao to a compatible version
!pip install --upgrade torchao

peft_model = get_peft_model(foundation_model, lora_config)

print(peft_model.print_trainable_parameters())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 55.3 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


trainable params: 98,304 || all params: 559,312,896 || trainable%: 0.0176
None


In [15]:
import transformers

# S'assurer que le trainer utilise le dataset 'data' mis à jour
trainer = transformers.Trainer(
    model=peft_model,
    train_dataset=data, # Utilise le dataset nettoyé avec padding/truncation
    args=transformers.TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=50,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=1,
        output_dir="outputs",
        remove_unused_columns=True, # Changé à True pour éviter de passer des colonnes non-tenseur
    ),
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

# Lancement de l'entraînement
trainer.train()

Step,Training Loss
1,0.000000
2,0.000000
3,0.000000
4,0.000000
5,0.000000
6,0.000000
7,0.000000
8,0.000000
9,0.000000
10,0.000000


TrainOutput(global_step=50, training_loss=0.85996178150177, metrics={'train_runtime': 21.2729, 'train_samples_per_second': 9.402, 'train_steps_per_second': 2.35, 'total_flos': 46450448793600.0, 'train_loss': 0.85996178150177, 'epoch': 0.796812749003984})

In [17]:
import time
import os

# Définition du répertoire de sortie
output_directory = "outputs"

time_now = int(time.time())

peft_model_path = os.path.join(
    output_directory,
    f"peft_model_{time_now}"
)

trainer.model.save_pretrained(peft_model_path)

In [18]:
from peft import PeftModel, PeftConfig

# Load the configuration and the model
config = PeftConfig.from_pretrained(peft_model_path)
model = AutoModelForCausalLM.from_pretrained(config.base_model_name_or_path)
model = PeftModel.from_pretrained(model, peft_model_path)

# Prepare a prompt
quote_prompt = "The best way to predict the future is"
inputs = tokenizer(quote_prompt, return_tensors="pt").to("cuda" if torch.cuda.is_available() else "cpu")

# Generate output
model.to(inputs['input_ids'].device)
with torch.no_grad():
    outputs = model.generate(input_ids=inputs["input_ids"], max_new_tokens=20)
    print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The best way to predict the future is to know the future before it happens.


In [19]:
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(model_name)

peft_model = PeftModel.from_pretrained(
    base_model,
    peft_model_path,
    is_trainable=False
)

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

In [20]:
inputs = tokenizer(
    "Two things are infinite: ",
    return_tensors="pt"
)

outputs = peft_model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_new_tokens=30,
    do_sample=True,
    temperature=0.7,
    top_p=0.9
)

print(tokenizer.batch_decode(outputs, skip_special_tokens=True))

['Two things are infinite:  the number of things in the universe and the number of events in the universe.']
